# Solution 20: Hybrid TTA on Solution 19

No retraining. I applied hybrid TTA to the Solution 19 adapter: single-pass for 2-choice questions (where TTA hurts) and full permutation TTA for 3/4/5-choice questions.

Based on validation analysis from Solution 15:
- 2-choice: single-pass (TTA hurts by -1.64%)
- 3-choice: TTA helps (+0.39%)
- 4-choice: TTA helps (+3.97%)
- 5-choice: TTA helps (+2.27%)

Matched Solution 15's score: 0.92354.

**Score: 0.92354**

## 0. Install Dependencies

In [1]:
!pip install -q "transformers==4.47.0"

!pip uninstall -y torchao 2>/dev/null

!pip install -q accelerate peft bitsandbytes datasets pillow tqdm pandas

import transformers, peft
print(f"transformers: {transformers.__version__}")
print(f"peft: {peft.__version__}")
print("All good. If first run, do Runtime -> Restart session, then run all cells.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 157.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 134.8 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.6 MB/s eta 0:00:00
transformers: 4.47.0
peft: 0.19.1
All good. If first run, do Runtime -> Restart session, then run all cells.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Imports & Configuration

In [3]:
import transformers
print(f"transformers: {transformers.__version__}")
assert transformers.__version__ >= "4.45.0", f"Too old: {transformers.__version__}"

import os, json, random, math
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from itertools import permutations

import torch
from transformers import AutoProcessor, AutoModelForVision2Seq
from peft import PeftModel

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
DATA_DIR = "/content/drive/MyDrive/pixels-to-predictions"
IMG_BASE = os.path.join(DATA_DIR, "images")
MAX_SEQ_LEN = 1024
IMAGE_LONGEST_EDGE = 512

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():


transformers: 4.47.0
Device: cuda


## 2. Load Data

In [4]:
val_df = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
print(f"Val: {len(val_df)} | Test: {len(test_df)}")

print(f"\nVal num_choices distribution:")
print(val_df['num_choices'].value_counts().sort_index())
print(f"\nTest num_choices distribution:")
print(test_df['num_choices'].value_counts().sort_index())

val_sp_passes = len(val_df[val_df['num_choices'] == 2])
val_tta_passes = sum(math.factorial(nc) for nc in val_df[val_df['num_choices'] >= 3]['num_choices'])
test_sp_passes = len(test_df[test_df['num_choices'] == 2])
test_tta_passes = sum(math.factorial(nc) for nc in test_df[test_df['num_choices'] >= 3]['num_choices'])
print(f"\nHybrid TTA forward passes:")
print(f"  Val: {val_sp_passes} (SP) + {val_tta_passes} (TTA) = {val_sp_passes + val_tta_passes}")
print(f"  Test: {test_sp_passes} (SP) + {test_tta_passes} (TTA) = {test_sp_passes + test_tta_passes}")

Val: 1048 | Test: 1008

Val num_choices distribution:
num_choices
2    244
3    508
4    252
5     44
Name: count, dtype: int64

Test num_choices distribution:
num_choices
2    272
3    438
4    260
5     38
Name: count, dtype: int64

Hybrid TTA forward passes:
  Val: 244 (SP) + 14376 (TTA) = 14620
  Test: 272 (SP) + 13428 (TTA) = 13700


## 3. Load Model & Processor (v19 adapter)

In [5]:
adapter_candidates = [
    "/content/smolvlm-lora-v19",
    "/content/drive/MyDrive/pixels-to-predictions/checkpoints_v19/epoch_6",
    "/content/drive/MyDrive/pixels-to-predictions/checkpoints_v19/epoch_5",
    "/content/drive/MyDrive/pixels-to-predictions/checkpoints_v19/lora_adapter",
]

adapter_path = None
for path in adapter_candidates:
    if os.path.exists(os.path.join(path, "adapter_config.json")):
        adapter_path = path
        break

if adapter_path is None:
    raise FileNotFoundError(f"No v19 adapter found! Checked: {adapter_candidates}")

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.image_processor.size = {"longest_edge": IMAGE_LONGEST_EDGE}

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

print(f"Loaded v19 adapter from: {adapter_path}")
print(f"Image size: {processor.image_processor.size}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Some kwargs in processor config are unused and will not have any effect: image_seq_len. 


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Loaded v19 adapter from: /content/drive/MyDrive/pixels-to-predictions/checkpoints_v19/epoch_6
Image size: {'longest_edge': 512}


## 4. Prediction Functions

In [6]:
def build_prompt(row, include_answer=False):
    """Identical to v16/v19."""
    choices = json.loads(row['choices'])
    choices_text = "\n".join(f"({i}) {c}" for i, c in enumerate(choices))

    parts = []
    if pd.notna(row.get('hint', None)) and str(row['hint']).strip():
        parts.append(f"Context: {row['hint'].strip()}")
    if pd.notna(row.get('lecture', None)) and str(row['lecture']).strip():
        parts.append(f"Background: {row['lecture'].strip()}")

    parts.append(f"Question: {row['question'].strip()}")
    parts.append(f"Choices:\n{choices_text}")
    parts.append("Answer with ONLY the number of the correct choice.")

    prompt_text = "\n\n".join(parts)
    if include_answer:
        return prompt_text, str(int(row['answer']))
    return prompt_text

DIGIT_TOKEN_IDS = [processor.tokenizer.encode(str(d), add_special_tokens=False)[0] for d in range(10)]
print(f"Digit token IDs: { {d: DIGIT_TOKEN_IDS[d] for d in range(5)} }")

@torch.no_grad()
def predict_single_pass(model, processor, df, img_base, batch_size=4):
    """Standard single-pass digit logit prediction."""
    model.eval()
    all_ids, all_preds = [], []

    for start in tqdm(range(0, len(df), batch_size), desc="Single-pass"):
        batch_df = df.iloc[start:start + batch_size]
        images, texts, batch_ids, nc_list = [], [], [], []

        for _, row in batch_df.iterrows():
            images.append(Image.open(os.path.join(img_base, row['image_path'])).convert("RGB"))
            prompt = build_prompt(row, include_answer=False)
            messages = [{"role": "user", "content": [
                {"type": "image"}, {"type": "text", "text": prompt}]}]
            texts.append(processor.apply_chat_template(messages, add_generation_prompt=True))
            batch_ids.append(row['id'])
            nc_list.append(row['num_choices'])

        inputs = processor(text=texts, images=images, return_tensors="pt",
                           padding=True, truncation=True, max_length=MAX_SEQ_LEN).to(device)

        with torch.amp.autocast("cuda", dtype=torch.float16):
            outputs = model(**inputs)
        logits = outputs.logits

        for i in range(len(batch_df)):
            last_pos = inputs["attention_mask"][i].sum().item() - 1
            next_logits = logits[i, last_pos, :]
            nc = nc_list[i]
            scores = torch.stack([next_logits[DIGIT_TOKEN_IDS[d]] for d in range(nc)])
            all_preds.append(scores.argmax().item())
            all_ids.append(batch_ids[i])

    return all_ids, all_preds

@torch.no_grad()
def predict_tta(model, processor, df, img_base, batch_size=4):
    """TTA: run all N! choice permutations, average logits mapped to original positions."""
    model.eval()
    all_ids, all_preds = [], []

    for idx in tqdm(range(len(df)), desc="TTA"):
        row = df.iloc[idx]
        nc = row['num_choices']
        choices = json.loads(row['choices'])
        all_perms = list(permutations(range(nc)))
        original_scores = torch.zeros(nc, device=device)
        image = Image.open(os.path.join(img_base, row['image_path'])).convert("RGB")

        for perm_start in range(0, len(all_perms), batch_size):
            perm_batch = all_perms[perm_start:perm_start + batch_size]
            images, texts, perms = [], [], []

            for perm in perm_batch:
                perm_row = row.copy()
                perm_row['choices'] = json.dumps([choices[perm[i]] for i in range(nc)])
                prompt = build_prompt(perm_row, include_answer=False)
                messages = [{"role": "user", "content": [
                    {"type": "image"}, {"type": "text", "text": prompt}]}]
                texts.append(processor.apply_chat_template(messages, add_generation_prompt=True))
                images.append(image.copy())
                perms.append(perm)

            inputs = processor(text=texts, images=images, return_tensors="pt",
                             padding=True, truncation=True, max_length=MAX_SEQ_LEN).to(device)

            with torch.amp.autocast("cuda", dtype=torch.float16):
                outputs = model(**inputs)
            logits = outputs.logits

            for i, perm in enumerate(perms):
                last_pos = inputs["attention_mask"][i].sum().item() - 1
                next_logits = logits[i, last_pos, :]
                digit_scores = torch.stack([next_logits[DIGIT_TOKEN_IDS[d]] for d in range(nc)])
                for d in range(nc):
                    original_scores[perm[d]] += digit_scores[d]

        pred = original_scores.argmax().item()
        all_preds.append(pred)
        all_ids.append(row['id'])

    return all_ids, all_preds

@torch.no_grad()
def predict_hybrid_tta(model, processor, df, img_base, batch_size=4):
    """Hybrid TTA: single-pass for 2-choice, full TTA for 3/4/5-choice."""
    model.eval()

    df_2 = df[df['num_choices'] == 2].reset_index(drop=True)
    df_345 = df[df['num_choices'] >= 3].reset_index(drop=True)

    print(f"  2-choice (single-pass): {len(df_2)} questions")
    print(f"  3/4/5-choice (TTA): {len(df_345)} questions")

    ids_2, preds_2 = [], []
    if len(df_2) > 0:
        ids_2, preds_2 = predict_single_pass(model, processor, df_2, img_base, batch_size)

    ids_345, preds_345 = [], []
    if len(df_345) > 0:
        ids_345, preds_345 = predict_tta(model, processor, df_345, img_base, batch_size)

    all_ids = ids_2 + ids_345
    all_preds = preds_2 + preds_345
    return all_ids, all_preds

print("Prediction functions defined.")

Digit token IDs: {0: 32, 1: 33, 2: 34, 3: 35, 4: 36}
Prediction functions defined.


## 5. Validation: Single-Pass vs Hybrid TTA

In [7]:
print("=== Single-Pass (v19) ===\n")
val_ids_sp, val_preds_sp = predict_single_pass(model, processor, val_df, IMG_BASE, batch_size=4)
val_acc_sp = np.mean([p == a for p, a in zip(val_preds_sp, val_df['answer'].tolist())])
print(f"Single-pass Val Accuracy: {val_acc_sp:.4f}")

print("\n=== Hybrid TTA (SP for 2-choice, TTA for 3/4/5) ===\n")
val_ids_hybrid, val_preds_hybrid = predict_hybrid_tta(model, processor, val_df, IMG_BASE, batch_size=4)

hybrid_pred_map = dict(zip(val_ids_hybrid, val_preds_hybrid))
val_preds_hybrid_aligned = [hybrid_pred_map[vid] for vid in val_df['id'].tolist()]
val_acc_hybrid = np.mean([p == a for p, a in zip(val_preds_hybrid_aligned, val_df['answer'].tolist())])
print(f"\nHybrid TTA Val Accuracy: {val_acc_hybrid:.4f}")

print(f"\n{'='*60}")
print(f"{'Method':<25} {'Val Acc':>10} {'vs v16 SP':>10} {'vs v16-TTA':>10}")
print(f"{'='*60}")
print(f"{'v16 Single-pass':<25} {'0.8979':>10} {'---':>10} {'---':>10}")
print(f"{'v16 TTA (Kaggle 0.923)':<25} {'0.9065':>10} {'+0.0086':>10} {'---':>10}")
print(f"{'v19 Single-pass':<25} {val_acc_sp:>10.4f} {val_acc_sp-0.8979:>+10.4f} {val_acc_sp-0.9065:>+10.4f}")
print(f"{'v19 Hybrid TTA':<25} {val_acc_hybrid:>10.4f} {val_acc_hybrid-0.8979:>+10.4f} {val_acc_hybrid-0.9065:>+10.4f}")
print(f"{'='*60}")

print(f"\nPer num_choices breakdown:")
sp_pred_map = dict(zip(val_ids_sp, val_preds_sp))
for nc in [2, 3, 4, 5]:
    mask = val_df['num_choices'] == nc
    true = val_df.loc[mask, 'answer'].tolist()
    ids_nc = val_df.loc[mask, 'id'].tolist()
    sp = [sp_pred_map[vid] for vid in ids_nc]
    hybrid = [hybrid_pred_map[vid] for vid in ids_nc]
    acc_sp = np.mean([p == a for p, a in zip(sp, true)])
    acc_hybrid = np.mean([p == a for p, a in zip(hybrid, true)])
    method = 'SP' if nc == 2 else 'TTA'
    print(f"  {nc}-choice ({method}): SP={acc_sp:.4f} Hybrid={acc_hybrid:.4f} delta={acc_hybrid-acc_sp:+.4f}")

=== Single-Pass (v19) ===



Single-pass:   0%|          | 0/262 [00:00<?, ?it/s]

Single-pass Val Accuracy: 0.9055

=== Hybrid TTA (SP for 2-choice, TTA for 3/4/5) ===

  2-choice (single-pass): 244 questions
  3/4/5-choice (TTA): 804 questions


Single-pass:   0%|          | 0/61 [00:00<?, ?it/s]

TTA:   0%|          | 0/804 [00:00<?, ?it/s]


Hybrid TTA Val Accuracy: 0.9084

Method                       Val Acc  vs v16 SP vs v16-TTA
v16 Single-pass               0.8979        ---        ---
v16 TTA (Kaggle 0.923)        0.9065    +0.0086        ---
v19 Single-pass               0.9055    +0.0076    -0.0010
v19 Hybrid TTA                0.9084    +0.0105    +0.0019

Per num_choices breakdown:
  2-choice (SP): SP=0.9016 Hybrid=0.9016 delta=+0.0000
  3-choice (TTA): SP=0.9390 Hybrid=0.9429 delta=+0.0039
  4-choice (TTA): SP=0.8929 Hybrid=0.8968 delta=+0.0040
  5-choice (TTA): SP=0.6136 Hybrid=0.6136 delta=+0.0000


## 6. Test Predictions (Hybrid TTA) + Submission

In [8]:
print("Generating test predictions with Hybrid TTA...")
test_ids_hybrid, test_preds_hybrid = predict_hybrid_tta(model, processor, test_df, IMG_BASE, batch_size=4)

print(f"\nPrediction distribution: {pd.Series(test_preds_hybrid).value_counts().sort_index().to_dict()}")

submission = pd.DataFrame({"id": test_ids_hybrid, "answer": test_preds_hybrid})
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
assert set(submission['id']) == set(sample_sub['id']), "ID mismatch!"
submission = submission.set_index('id').loc[sample_sub['id']].reset_index()
submission.to_csv("submission.csv", index=False)
print("\nSaved submission.csv (v19 Hybrid TTA)")
print(submission.head(10))

Generating test predictions with Hybrid TTA...
  2-choice (single-pass): 272 questions
  3/4/5-choice (TTA): 736 questions


Single-pass:   0%|          | 0/68 [00:00<?, ?it/s]

TTA:   0%|          | 0/736 [00:00<?, ?it/s]


Prediction distribution: {0: 357, 1: 354, 2: 219, 3: 75, 4: 3}

Saved submission.csv (v19 Hybrid TTA)
           id  answer
0  test_01750       2
1  test_00128       0
2  test_02891       3
3  test_02425       1
4  test_00930       2
5  test_03725       2
6  test_00009       1
7  test_02880       0
8  test_01208       0
9  test_00619       1


## 7. Download

In [9]:
from google.colab import files
files.download('submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>